# Experiment 3: Precision-Recall Trade-off Under Construct-Similar vs. Construct-Diverse Stacking (Testing H3)


In [2]:
!pip install -q transformers datasets torch scikit-learn pandas accelerate

In [1]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import recall_score, precision_score, f1_score, confusion_matrix, roc_curve
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Setup] Device: {device}")

REFUSAL_PREFIXES = ("i cannot", "i'm sorry", "i am sorry", "i am unable", "as an ai", "i will not", "i can't")
TARGET_FPR = 0.05

# ==============================================================================
# 1. CURATE UNIFIED EVALUATION CORPUS
# ==============================================================================
wg_test = load_dataset("allenai/wildguardmix", "wildguardtest", split="test")
wg_train = load_dataset("allenai/wildguardmix", "wildguardtrain", split="train")
df_wg = pd.concat([pd.DataFrame(wg_test), pd.DataFrame(wg_train)], ignore_index=True)

harm_col = "is_response_harmful" if "is_response_harmful" in df_wg.columns else "response_harm_label"
df_wg["_label"] = df_wg[harm_col].apply(lambda x: 1 if str(x).strip().lower() in ("harmful", "true", "1") else 0)
df_wg["_completion"] = df_wg["response"].fillna("")
df_wg["_prompt"] = df_wg["prompt"]

is_refusal = df_wg["_completion"].str.lower().str.startswith(REFUSAL_PREFIXES)
df_wg_harmful = df_wg[(df_wg["_label"] == 1) & (~is_refusal)].sample(n=1000, random_state=42)
df_wg_safe = df_wg[df_wg["_label"] == 0].sample(n=500, random_state=42)

df_wg_harmful["_group"] = "wildguard_harmful"
df_wg_safe["_group"] = "wildguard_safe"

xs_data = load_dataset("allenai/xstest-response", split="response_harmfulness")
df_xs = pd.DataFrame(xs_data)

def is_safe(x):
    return str(x).strip().lower() in ("safe", "false", "0", "unharmful", "benign")

p_col = "prompt" if "prompt" in df_xs.columns else "user_input"
r_col = "response" if "response" in df_xs.columns else "model_output"
l_col = "label" if "label" in df_xs.columns else "response_harmfulness"

df_xs["_is_safe"] = df_xs[l_col].apply(is_safe)
df_xs_safe = df_xs[df_xs["_is_safe"]].copy()

if len(df_xs_safe) == 0:
    df_xs_safe = df_xs[df_xs[l_col].astype(str).str.lower() != "harmful"].copy()

df_xs_safe = df_xs_safe.rename(columns={p_col: "_prompt", r_col: "_completion"})
df_xs_safe["_label"] = 0
df_xs_safe["_group"] = "xstest_safe_sounding"

need_replace = len(df_xs_safe) < 500
df_xs_safe = df_xs_safe.sample(n=500, replace=need_replace, random_state=42).reset_index(drop=True)

corpus = pd.concat([df_wg_harmful, df_wg_safe, df_xs_safe], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"[Corpus Check] Harmful: {sum(corpus['_label']==1)} | Safe: {sum(corpus['_label']==0)}")

# ==============================================================================
# 2. RUN INFERENCE ON FULL CORPUS (Populates Raw Score Columns)
# ==============================================================================
tok_a1 = AutoTokenizer.from_pretrained("unitary/toxic-bert")
mod_a1 = AutoModelForSequenceClassification.from_pretrained("unitary/toxic-bert").to(device)

tok_a2 = AutoTokenizer.from_pretrained("s-nlp/roberta_toxicity_classifier")
mod_a2 = AutoModelForSequenceClassification.from_pretrained("s-nlp/roberta_toxicity_classifier").to(device)

tok_b = AutoTokenizer.from_pretrained("OpenAssistant/reward-model-deberta-v3-large-v2")
mod_b = AutoModelForSequenceClassification.from_pretrained("OpenAssistant/reward-model-deberta-v3-large-v2").to(device)

def get_scores_a(texts, tok, mod):
    mod.eval()
    scores = []
    for i in range(0, len(texts), 32):
        batch = [str(t) for t in texts[i:i+32]]
        inputs = tok(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
        with torch.no_grad():
            probs = torch.sigmoid(mod(**inputs).logits)
            s = torch.max(probs, dim=1)[0] if probs.shape[1] > 1 else probs.squeeze(-1)
        scores.extend(s.cpu().numpy())
    return np.array(scores)

def get_scores_b(prompts, completions, tok, mod):
    mod.eval()
    formatted = [f"User: {p}\nAssistant: {c}" for p, c in zip(prompts, completions)]
    scores = []
    for i in range(0, len(formatted), 16):
        batch = formatted[i:i+16]
        inputs = tok(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
        with torch.no_grad():
            rewards = mod(**inputs).logits.squeeze(-1)
        scores.extend(rewards.cpu().numpy())
    return np.array(scores)

print("Scoring Model A1 (toxic-bert)...")
corpus["raw_A1"] = get_scores_a(corpus["_completion"].tolist(), tok_a1, mod_a1)

print("Scoring Model A2 (roberta-toxicity)...")
corpus["raw_A2"] = get_scores_a(corpus["_completion"].tolist(), tok_a2, mod_a2)

print("Scoring Model B (reward-model)...")
corpus["raw_B"]  = get_scores_b(corpus["_prompt"].tolist(), corpus["_completion"].tolist(), tok_b, mod_b)

# ==============================================================================
# 3. SPLIT CORPUS & CALIBRATE THRESHOLDS
# ==============================================================================
# Create joint stratification key and split AFTER scoring
strat_key = corpus["_label"].astype(str) + "_" + corpus["_group"]
calib_idx, eval_idx = train_test_split(corpus.index, test_size=0.6, stratify=strat_key, random_state=42)

calib_df = corpus.loc[calib_idx].copy()
eval_df = corpus.loc[eval_idx].copy()

print(f"[Split Check] Calibration Rows: {len(calib_df)} | Evaluation Rows: {len(eval_df)}")

def calibrate_roc(labels, raw_scores, target_fpr=TARGET_FPR, higher_means_harm=True):
    scores = raw_scores if higher_means_harm else -raw_scores
    fpr_curve, tpr_curve, thresholds = roc_curve(labels, scores)
    idx = np.argmin(np.abs(fpr_curve - target_fpr))
    chosen = thresholds[idx]
    return chosen if higher_means_harm else -chosen

# Now calib_df exists and contains "raw_A1", "raw_A2", and "raw_B"
tau_a1 = calibrate_roc(calib_df["_label"].values, calib_df["raw_A1"].values, TARGET_FPR, True)
tau_a2 = calibrate_roc(calib_df["_label"].values, calib_df["raw_A2"].values, TARGET_FPR, True)
tau_b  = calibrate_roc(calib_df["_label"].values, calib_df["raw_B"].values,  TARGET_FPR, False)

print(f"Calibrated Thresholds -> tau_A1: {tau_a1:.4f} | tau_A2: {tau_a2:.4f} | tau_B: {tau_b:.4f}")

[Setup] Device: cuda


README.md:   0%|          | 0.00/6.16k [00:00<?, ?B/s]

test/wildguard_test.parquet: reconstructing file:   0%|          |  0.00B / 2.26MB            

test/wildguard_test.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

...

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

Scoring Model A1 (toxic-bert)...


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.74GB            

model.safetensors: downloading bytes:           |  0.00B            

Scoring Model A2 (roberta-toxicity)...
Scoring Model B (reward-model)...
[Split Check] Calibration Rows: 800 | Evaluation Rows: 1200
Calibrated Thresholds -> tau_A1: 0.0121 | tau_A2: 0.9934 | tau_B: -4.2377


In [2]:
# ==============================================================================
# Corrected scoring for BINARY toxic classifiers (e.g. roberta_toxicity_classifier)
# ==============================================================================

def get_scores_binary_toxic_class(texts, tok, mod, toxic_label_index=1):
    """
    For 2-class toxicity classifiers (neutral, toxic). Returns P(toxic)
    specifically, NOT max(P(neutral), P(toxic)) - the latter is close to 1.0
    for almost every input regardless of class, since the two sum to 1,
    which is why tau_A2 was calibrating to a near-ceiling value.
    """
    mod.eval()
    scores = []
    for i in range(0, len(texts), 32):
        batch = [str(t) for t in texts[i:i + 32]]
        inputs = tok(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
        with torch.no_grad():
            probs = torch.softmax(mod(**inputs).logits, dim=-1)
        scores.extend(probs[:, toxic_label_index].cpu().numpy())
    return np.array(scores)


print("Re-scoring Model A2 (roberta-toxicity) with corrected binary-class scoring...")
corpus["raw_A2"] = get_scores_binary_toxic_class(corpus["_completion"].tolist(), tok_a2, mod_a2)

# Re-split isn't needed since calib_idx/eval_idx don't depend on raw_A2 values,
# but calib_df/eval_df are copies made BEFORE this re-scoring, so refresh them:
calib_df = corpus.loc[calib_idx].copy()
eval_df = corpus.loc[eval_idx].copy()

# Re-calibrate tau_a2 with the corrected scores
tau_a2 = calibrate_roc(calib_df["_label"].values, calib_df["raw_A2"].values, TARGET_FPR, True)
print(f"Corrected tau_A2: {tau_a2:.4f}  (compare to the old, near-ceiling 0.9934)")


# ==============================================================================
# SECTION 3  EVALUATION & STATISTICAL COMPARISON
# ==============================================================================
from sklearn.metrics import recall_score, precision_score, f1_score, confusion_matrix

eval_df["f_A1"] = (eval_df["raw_A1"] > tau_a1).astype(int)
eval_df["f_A2"] = (eval_df["raw_A2"] > tau_a2).astype(int)
eval_df["f_B"] = (eval_df["raw_B"] < tau_b).astype(int)

eval_df["F_mono"] = eval_df["f_A1"]
eval_df["F_sim2"] = eval_df["f_A1"] | eval_df["f_A2"]
eval_df["F_div2"] = eval_df["f_A1"] | eval_df["f_B"]

y_true = eval_df["_label"].values


def compute_metrics(preds, y):
    rec = recall_score(y, preds)
    prec = precision_score(y, preds, zero_division=0)
    f1 = f1_score(y, preds, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, preds).ravel()
    fpr = fp / (fp + tn)
    return rec, prec, fpr, f1


m_mono = compute_metrics(eval_df["F_mono"].values, y_true)
m_sim2 = compute_metrics(eval_df["F_sim2"].values, y_true)
m_div2 = compute_metrics(eval_df["F_div2"].values, y_true)

delta_f1_sim2 = m_sim2[3] - m_mono[3]
delta_f1_div2 = m_div2[3] - m_mono[3]
diff_of_deltas = delta_f1_div2 - delta_f1_sim2

print("\n" + "=" * 80)
print("H3 STATISTICAL EVALUATION SUMMARY")
print("=" * 80)
print(f"Topology 0 [F_mono]: Recall={m_mono[0]:.4f} | Prec={m_mono[1]:.4f} | FPR={m_mono[2]:.4f} | F1={m_mono[3]:.4f}")
print(f"Topology 1 [F_sim2]: Recall={m_sim2[0]:.4f} | Prec={m_sim2[1]:.4f} | FPR={m_sim2[2]:.4f} | F1={m_sim2[3]:.4f} | ΔF1={delta_f1_sim2:+.4f}")
print(f"Topology 2 [F_div2]: Recall={m_div2[0]:.4f} | Prec={m_div2[1]:.4f} | FPR={m_div2[2]:.4f} | F1={m_div2[3]:.4f} | ΔF1={delta_f1_div2:+.4f}")
print("-" * 80)
print(f"Observed Advantage of Diversity over Similarity (ΔF1,div2 - ΔF1,sim2): {diff_of_deltas:+.4f}")

# Per-source FPR breakdown by topology
print("\nFalse-positive rate by safe-control source, per topology:")
for group in ["wildguard_safe", "xstest_safe_sounding"]:
    sub = eval_df[eval_df["_group"] == group]
    if len(sub) == 0:
        continue
    for topo_name, col in [("F_mono", "F_mono"), ("F_sim2", "F_sim2"), ("F_div2", "F_div2")]:
        fpr = sub[col].mean()
        print(f"  {group:22s} {topo_name:8s} n={len(sub):4d}  FPR={fpr:.4f}")

# Paired bootstrap on the difference-of-deltas
np.random.seed(42)
boot_diff_deltas = []
boot_div2_mono = []

for _ in range(10000):
    idx = np.random.choice(len(y_true), size=len(y_true), replace=True)
    y_b = y_true[idx]
    f1_m = f1_score(y_b, eval_df["F_mono"].values[idx], zero_division=0)
    f1_s = f1_score(y_b, eval_df["F_sim2"].values[idx], zero_division=0)
    f1_d = f1_score(y_b, eval_df["F_div2"].values[idx], zero_division=0)

    d_sim = f1_s - f1_m
    d_div = f1_d - f1_m

    boot_div2_mono.append(d_div)
    boot_diff_deltas.append(d_div - d_sim)

ci_div_mono = np.percentile(boot_div2_mono, [2.5, 97.5])
ci_diff_deltas = np.percentile(boot_diff_deltas, [2.5, 97.5])
frac_favorable = np.mean(np.array(boot_diff_deltas) > 0)

print(f"\n95% CI for ΔF1,div2 (F_div2 - F_mono): [{ci_div_mono[0]:+.4f}, {ci_div_mono[1]:+.4f}]")
print(f"95% CI for Direct Advantage (ΔF1,div2 - ΔF1,sim2): [{ci_diff_deltas[0]:+.4f}, {ci_diff_deltas[1]:+.4f}]")
print(f"Fraction of Bootstrap Resamples Favoring Diversity (Diff > 0): {frac_favorable:.3f}")
print("=" * 80)

if ci_diff_deltas[0] > 0:
    print("VERDICT: [H3 CONFIRMED] Construct diversity yields a statistically significant "
          "F1 advantage over construct similarity (95% CI excludes zero).")
else:
    print("VERDICT: [H3 INCONCLUSIVE] The F1 advantage of construct diversity over similarity "
          "is not statistically distinguishable from noise at this sample size.")

Re-scoring Model A2 (roberta-toxicity) with corrected binary-class scoring...
Corrected tau_A2: 0.0006  (compare to the old, near-ceiling 0.9934)

H3 STATISTICAL EVALUATION SUMMARY
Topology 0 [F_mono]: Recall=0.2250 | Prec=0.8182 | FPR=0.0500 | F1=0.3529
Topology 1 [F_sim2]: Recall=0.2783 | Prec=0.8068 | FPR=0.0667 | F1=0.4139 | ΔF1=+0.0609
Topology 2 [F_div2]: Recall=0.4050 | Prec=0.7915 | FPR=0.1067 | F1=0.5358 | ΔF1=+0.1829
--------------------------------------------------------------------------------
Observed Advantage of Diversity over Similarity (ΔF1,div2 - ΔF1,sim2): +0.1220

False-positive rate by safe-control source, per topology:
  wildguard_safe         F_mono   n= 300  FPR=0.0133
  wildguard_safe         F_sim2   n= 300  FPR=0.0167
  wildguard_safe         F_div2   n= 300  FPR=0.1233
  xstest_safe_sounding   F_mono   n= 300  FPR=0.0867
  xstest_safe_sounding   F_sim2   n= 300  FPR=0.1167
  xstest_safe_sounding   F_div2   n= 300  FPR=0.0900

95% CI for ΔF1,div2 (F_div2 - F